Replay mode uses recorded fixtures and does not call a live model. Set `NORTHSTAR_MODE=live` with `GEMINI_API_KEY` to opt in, or use `NORTHSTAR_MODE=record` to save synthetic responses.

In [ ]:
from pathlib import Path
from northstar.runtime import get_client, PromptRequest, Message, HASH_EMBEDDING_NOTICE
from lab03 import run_lab, select_examples, EXAMPLE_BANK

client = get_client(Path("fixtures/replays.json"))  # NORTHSTAR_MODE=replay|live|record

# 03 — Constraints, Examples, and Few-Shot Learning

## Scenario

Northstar routes support messages to a small set of categories. This lab compares four example-selection strategies on a frozen evaluation set.

## Baseline: Zero-Shot

We start with a direct instruction and no examples.

In [ ]:
metrics = run_lab(client)
assert metrics["zero_accuracy"].numerator == 3
assert metrics["zero_accuracy"].denominator == 5

## Strategy 1: Static Few-Shot

Fixed examples are predictable but cost context tokens on every request.

In [ ]:
assert metrics["static_accuracy"].numerator == 4

## Strategy 2: Random Few-Shot Selection

Selection uses `random.Random(case_id)`, not process-randomized hashes.

In [ ]:
assert metrics["random_accuracy"].numerator == 3

## Strategy 3: Similarity Selection

Offline similarity is lexical-hash, not semantic; ordering may differ live.

In [ ]:
print(HASH_EMBEDDING_NOTICE)
assert metrics["similarity_accuracy"].numerator == 5
assert metrics["similarity_accuracy"].numerator >= metrics["static_accuracy"].numerator >= metrics["zero_accuracy"].numerator
query = EXAMPLE_BANK[0]["message"]
assert query not in {item["message"] for item in select_examples(2, query, EXAMPLE_BANK, "b03/notebook/leakage")}

## Takeaway

Examples should target measured boundaries, and retrieval complexity must earn its token cost.

## References

- [Core concepts and workflow](README.md#core-concepts--workflow)
- [Production best practices](README.md#production-best-practices)